In [ ]:
## Imports ##

# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Misc Data Handling #
import pandas as pd
import time
import re
import json 
from datetime import date 



# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
from itertools import chain
import ast
import random

# Flight Trajectory Functions #
from dsci_550_a1.flightFunctions import *

# Plotly #
import plotly.graph_objects as go
import plotly.express as px
import textwrap


In [165]:
# Helper Functions #
from dsci_550_a3.dg_viz import hp_interactive_globe
from dsci_550_a3.dg_query import filter_hp_df, get_legend_items
from dsci_550_a3.dg_dataLoader import load_all_data

# Pandas
import pandas as pd

# Parsing Date
from datetime import date
import ast
import json

# Preparing Haunted Place Images
import os
import base64


def prepare_hover_image(encoded_image):
    if pd.isna(encoded_image) or encoded_image == "":
        return "<b>No image available</b>"
    else:
        return f"<img src='{encoded_image}' width='120' height='80'>"

# Point to image
def encode_image(image_pointer, img_directory):

    # If there is no pointer, use default image
    if pd.isna(image_pointer) or image_pointer == "":
        img_path = os.path.join(img_directory, 'hpimg_placeholder.png')
    else:
        img_path = os.path.join(img_directory, 'hpimg_placeholder.png')
        #img_path = os.path.join(img_directory, image_pointer)
    
    # if file does not exist, use default
    if not os.path.exists(img_path):
        img_path = os.path.join(img_directory, 'hpimg_placeholder.png')
    
    # open file and encode
    with open(img_path, 'rb') as f:
        encoded = base64.b64encode(f.read()).decode()

    return 'data:image/png;base64,{}'.format(encoded)


# Encode image date for hover info
# def encode_image(image_pointer, img_directory):
#     try:
#         with open(image_pointer, 'rb') as f:
#             encoded = base64.b64encode(f.read()).decode()
#     except FileNotFoundError:
#         with open(prepare_hover_image('hpimg_placeholder.png', img_directory), 'rb') as f:
#             encoded = base64.b64encode(f.read()).decode()
#     return 'data:image/png;base64,{}'.format(encoded)

# Date parsing helper
def parse_date(s): 
    return date(*map(int, s.split('-'))) 

# Load Data
def load_all_data(img_directory):
    hp_df = pd.read_csv("../data/processed/haunted_places_features_added_v2.tab", sep="\t")
    hp_df['Haunted_Places_Date'] = hp_df['Haunted_Places_Date'].apply(lambda x: ast.literal_eval(x)) 
    hp_df['Haunted_Places_Date'] = hp_df['Haunted_Places_Date'].apply(lambda x: [parse_date(y) for y in x] if isinstance(x, list) else x)
    
    # Encode hpimg data
    hp_df['Image_Pointer'] = hp_df['Image_Pointer'].fillna('hpimg_placeholder.png')
    hp_df['Image_Pointer'] = hp_df['Image_Pointer'].apply(lambda x: encode_image(x, img_directory))

    route_df = pd.read_csv("../data/joined_datasets/american_routes.tsv", sep="\t")
    route_df["Flight_Path"] = route_df["Flight_Path"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    airport_df = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep="\t")
    airport_df["Airport_Radius"] = airport_df["Airport_Radius"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

    with open("../data/processed/flight_proximity_data.json", "r") as f:
        flight_intersections = json.load(f)

    with open("../data/processed/airport_proximity_data.json", "r") as f:
        airport_intersections = json.load(f)

    return (
        hp_df,
        route_df,
        airport_df,
        flight_intersections,
        airport_intersections
    )

hpimg_dir = "../data/generated_images"

## Load Data and Define Holidays
(
    hp_df,
    route_df,
    airport_df,
    flight_intersections,
    airport_intersections
) = load_all_data(hpimg_dir)

hp_df['Longitude'].dtype

dtype('float64')

In [186]:
from itertools import chain
from datetime import date

# Unique Legend Items
def get_legend_items(df_hp, legend_key):
    
    # return True and False if Bool
    if df_hp[legend_key].dtype == 'bool':
        return ['True', 'False']
    
    s = set().union(*df_hp[legend_key].dropna().str.split(' | ').tolist())
    try:
        s.remove('|')  # remove delimiter if it was caught
    except:
        pass
    return  sorted(list(s))


# Date Range Filter
def parse_date(s): 
    return date(*map(int, s.split('-'))) 

def convert_date_str(date):
    return date.strftime('%Y-%m-%d')

def in_date_range(date_list, start_date, end_date):
    return any(start_date <= date <= end_date for date in date_list)

def query_df(query_keys, s):

    # Conver to list if single string is passed
    if isinstance(query_keys, str):
        query_keys = [query_keys]

    # Return False if df is null 
    if pd.isna(s) or s is None:
        return False
    # Make logical "or" regex and query
    query_regex = "|".join(map(re.escape, query_keys))
    return bool(re.search(query_regex, s))

def filter_hp_df(
    hp_df,
    route_df,
    airport_df,
    flight_intersections,
    airport_intersections,
    state=None, event_type=None, apparition_type=None, haunt_date_range=None, holiday = None):
    
    filtered_hp_df = hp_df.copy()

    if state:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['State'].apply(lambda s: query_df(state, s))]
    if event_type:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Event_Type'].apply(lambda s: query_df(event_type, s))]
    if apparition_type:
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Apparition_Type'].apply(lambda s: query_df(apparition_type, s))]
    if haunt_date_range:
        start_date, end_date = map(parse_date, haunt_date_range)
        filtered_hp_df = filtered_hp_df[(filtered_hp_df['Haunted_Places_Date'].apply(lambda x: in_date_range(x, start_date, end_date)))]
    if holiday:
        holiday = parse_date(holiday)
        filtered_hp_df = filtered_hp_df[filtered_hp_df['Haunted_Places_Date'].apply(lambda x: in_date_range(x, holiday, holiday))]
    
    filtered_hp_df['Haunted_Places_Date'] = filtered_hp_df['Haunted_Places_Date'].apply(lambda x: [convert_date_str(y) for y in x])
    
    filtered_hp_df = filtered_hp_df.astype(str)

    haunted_ids = filtered_hp_df['Haunted_Places_Id'].tolist()

    filtered_flight_intersections = {k: v['Routes'] for k, v in flight_intersections.items() if k in haunted_ids}
    filtered_airport_intersections = {k: v['Airports'] for k, v in airport_intersections.items() if k in haunted_ids}

    relevant_routes = set()
    relevant_iata_codes = set()
    relevant_airports = set()

    for _, v in filtered_flight_intersections.items():
        relevant_routes.update(route['Route_ID'] for route in v)
        relevant_iata_codes.update(
            chain(
            (route['Dest_Airport'] for route in v),
            (route['Source_Airport'] for route in v)
            )
        )
        
    for _, v in filtered_airport_intersections.items():
        relevant_airports.update(airport['Airport_ID'] for airport in v)

    filtered_route_df = route_df.loc[list(relevant_routes)]
    filtered_airport_df = airport_df[airport_df['Id'].isin(relevant_airports) | airport_df['Iata_Code'].isin(relevant_iata_codes)]
    
    filtered_hp_df = filtered_hp_df.astype(str)
    
    return filtered_hp_df, filtered_route_df, filtered_airport_df




legend_arg = None
state = 'Michigan'
event_type = ['Plane_Crash', 'Electronic_Malfunction']
apparition_type = None
haunt_date_range = None
holiday = None


if not legend_arg:
    legend_arg = {'Event_Type': get_legend_items(hp_df, 'Event_Type')} 

filtered_hp_df, filtered_route_df, filtered_airport_df = filter_hp_df(
    hp_df,
    route_df,
    airport_df,
    flight_intersections,
    airport_intersections,
    state,
    event_type,
    apparition_type,
    haunt_date_range,
    holiday,
)

filtered_hp_df


,Haunted_Places_Id,City,Country,Description,Location,State,State_Abbrev,Longitude,Latitude,City_Longitude,...,Distance_to_Nearest_Worship,Religion_Intersection,Daylight_Duration_Hours,Named_Entities,Image_Pointer,Image_Caption,Image_Objects,GeoTopic_Locations,GeoTopic_Latitudes,GeoTopic_Longitudes
53,53,Canton,United States,The story goes that a group of kids were playi...,Denton Road Bridge,Michigan,MI,-83.52522599999999,42.2853235,-83.48211599999999,...,3923.27,christian,10.402,"[('Denton Road', 'FAC'), ('one', 'CARDINAL'), ...","data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",a black and white photo of a boat in the water .,"boathouse: 74.76%, viaduct: 1.37%, church: 1.3...","Canton de Genève, Michigan","46.19673, 44.25029","6.11044, -85.50033"
61,61,Cheboygan,United States,An old farmer went insane and killed his famil...,Hikyes Tomb,Michigan,MI,-84.4744795,45.64695630000001,-84.4744795,...,156.5,nan,8.841,"[('night', 'TIME'), ('2', 'CARDINAL')]","data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",a wooden bench sitting in front of a window .,"buckle: 8.82%, park_bench: 5.68%, studio_couch...","Cheboygan County, Michigan","45.47294, 44.25029","-84.49206, -85.50033"
97,97,Dearborn Heights,United States,Computer number 25 is definitely haunted by an...,Crestwood Computer Lab,Michigan,MI,-83.2932414,42.3220925,-83.27326269999999,...,1203.73,christian,9.799,"[('number 25', 'CARDINAL')]","data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",a black and white photo of a cat sitting on a ...,"rocking_chair: 52.04%, barber_chair: 18.63%, t...",Dearborn County,39.14519,-84.97326
287,287,Marine City,United States,Once in a great while employees of the kitchen...,Riviera Restaurant,Michigan,MI,-82.49271159999999,42.7146215,-82.492132,...,1766.98,nan,9.45,"[('the summer of 2003', 'DATE'), ('5-10 minute...","data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",a black and white photo of a toaster oven,"radio: 21.45%, cassette_player: 13.59%, mailbo...",Marine City,42.71948,-82.49213
313,313,Milan,United States,Legends have it that on Milan Oakvill Rd. In 1...,Milan Oakvill Rd.,Michigan,MI,-83.62685549999999,42.0841868,-83.6824384,...,7121.54,christian,9.23,"[('Milan Oakvill Rd', 'FAC'), ('1910', 'DATE')...","data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",a bench sitting in the middle of a forest .,"barn: 36.97%, park_bench: 13.76%, church: 11.6...",Milan,45.46427,9.18951
522,522,Westland,United States,There is a small hill toward the center of the...,Westland Meadows Trailer Park,Michigan,MI,-83.340879,42.270212,-83.400211,...,1157.58,christian,9.21,"[('around the 1800', 'DATE'), ('the Eloise Ins...","data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAA...",a black and white photo of a park bench,"stone_wall: 23.46%, fire_screen: 6.35%, patio:...","Michigan, City of Walnut Grove","44.25029, 33.7454","-85.50033, -83.85027"


In [ ]:
def generate_formatted_textbox(row, primary_tag, additional_tags, separator="|"):
    """
    Generate a formatted HTML string for a row based on specified fields.

    Parameters:
    - row: A pandas Series (a single row of a DataFrame).
    - fields: List of fields (column names) to include.
    - bold_fields: List of fields to bold (optional; if None, all fields will be bolded).
    - separator: String separator between lines (default: "<br>").
    - extra_formatting: Optional dictionary {field: custom_format_string}, where
                        custom_format_string can use {value} as placeholder.

    Returns:
    - A formatted HTML string.
    """
    lines = []
    lines.append(f"<b>Location</b>: {row['Location']} | <b>{primary_tag}</b>: {row.get(primary_tag, "")}")

    additional_tag_line = ['<b>Additional Tags:</b>'] + [f"{row.get(tag, '')}" for tag in additional_tags]
    lines.append(f" {separator} ".join(additional_tag_line))

    lines.append(f"<b>Number of Intersecting Flights</b>: {row['Flight_Intersection_Count']} | <b>Number of Nearby Airports</b>: {row['Aerodrome_Count']}<br>")
    lines.append(f"<b>Description</b>: {row['Formatted_Description']}")

    return "<br>".join(lines)





In [ ]:
## Imports ##

# System Path #
import os
import sys 

# Misc Data Handling #
import pandas as pd
from datetime import date 

# Flight Trajectory Functions #
from dsci_550_a1.flightFunctions import *

# Plotly #
import plotly.graph_objects as go
import plotly.express as px
import textwrap


def prepare_hover_image(encoded_image):
    if pd.isna(encoded_image) or encoded_image == "":
        return "<b>No image available</b>"
    else:
        return f"<img src='{encoded_image}' width='120' height='80'>"
    

## Main Visualization Function ##
def hp_interactive_globe(hp_df, route_df, airport_df, legend_arg, additional_tags = None):
    # Initialize trace lists
    all_traces = []

    legend_key = list(legend_arg.keys())[0]
    legend_values = [v for v in legend_arg[legend_key]]

    # Color Palette
    plot_colors = {
    'dsci550_a3-1-hsla': 'rgb(116, 84, 190)', 'dsci550_a3-2-hsla': 'rgb(125, 133, 241)', 'dsci550_a3-3-hsla': 'rgb(58, 45, 113)', 'dsci550_a3-4-hsla': 'rgb(10, 188, 4)', 'dsci550_a3-5-hsla': 'rgb(77, 114, 23)', 
    'dsci550_a3-6-hsla': 'rgb(140, 215, 64)', 'dsci550_a3-7-hsla': 'rgb(237, 7, 7)', 'dsci550_a3-8-hsla': 'rgb(113, 3, 3)', 'dsci550_a3-9-hsla': 'rgb(188, 53, 4)',  'dsci550_a3-10-hsla': 'rgb(238, 114, 6)', 
    'dsci550_a3-11-hsla': 'rgb(241, 166, 74)', 'dsci550_a3-12-hsla': 'rgb(255, 137, 254)', 'dsci550_a3-13-hsla': 'rgb(116, 235, 148)', 'dsci550_a3-14-hsla': 'rgb(4, 214, 176)', 'dsci550_a3-15-hsla': 'rgb(65, 173, 240)', 
    'dsci550_a3-16-hsla': 'rgb(2, 100, 109)', 'dsci550_a3-17-hsla': 'rgb(1, 38, 59)', 'dsci550_a3-18-hsla': 'rgb(191, 176, 88)', 'dsci550_a3-19-hsla': 'rgb(34, 3, 1)', 'dsci550_a3-20-hsla': 'rgb(12, 12, 12)'
    }
        

    ## Haunted Places Trace 
    
    # Iterate through legend values
    for i, val in enumerate(legend_values):
        
        # Filter Dataset
        hp_df_filtered = hp_df.loc[hp_df[f'{legend_key}'].str.contains(val, na=False)].copy()
        hp_df_filtered['Formatted_Description'] = hp_df_filtered['Description'].apply(
        lambda x: "<br>".join(textwrap.wrap(x, width=50))
    )
        # Add Trace
        trace = (go.Scattergeo(
            locationmode = 'USA-states',
            lon = hp_df_filtered['Longitude'],
            lat = hp_df_filtered['Latitude'],
            hoverinfo = 'skip',
            customdata = np.stack([
                                hp_df_filtered.apply(lambda row: generate_formatted_textbox(
                                                row,
                                                primary_tag=f"{legend_key}",
                                                additional_tags=additional_tags,
                                            ), 
                                        axis=1),
            # customdata = np.stack([
            #                     hp_df_filtered.apply(lambda row: 
            #                             f"<b>Location</b>: {row['Location']} | <b>{legend_key}</b>: {row[f'{legend_key}']}<br>"
            #                             f"<b>Additional Tags</b>: {row['Haunted_Places_Id']} | <b>Location</b>: {row['Location']}<br>"
            #                             f"<b>Number of Intersecting Flights</b>: {row['Flight_Intersection_Count']} | <b>Number of Nearby Airports</b>: {row['Aerodrome_Count']}<br>"
            #                             f"<b>Description</b>: {row['Formatted_Description']}", 
            #                             axis=1),
                                ], axis = 1
            ),
            hovertemplate = (
                "%{customdata[0]}<br><br>"
            ),
            mode = 'markers',
            showlegend = True, 
            marker = dict(
                size = 4,
                color = plot_colors[f"dsci550_a3-{i+1}-hsla"],
                opacity = 0.75
                ),
                name = val, 
                visible = True
            )
        )
        all_traces.append(trace)


    ## Flight Paths Traces

    lats_plot, lons_plot = [] , []

    for row in route_df.itertuples(index = False):   

        lats, lons = zip(*row.Flight_Path)
        lats, lons = list(lats), list(lons)

        lats_plot.extend(lats + [None])
        lons_plot.extend(lons + [None])

    # Add trace
    trace = (go.Scattergeo(
        lon= lons_plot,
        lat= lats_plot,
        mode='lines',
        line=dict(width=.5, color='red'),
        opacity = 0.2, 
        hoverinfo = 'skip', 
        name = "Flights",
        visible = False
    ))
    all_traces.append(trace)


    ## Airport Traces
    airport_types = airport_df['Type'].unique().tolist()

    # Bluescale Color Palette
    airport_plot_colors = {
    'heliport' :        "rgb(100,151,177)" ,
    'seaplane_base': 	"rgb(179,205,224)",
    'balloonport' : 	"rgb(179,205,224)",
    'small_airport' :  "rgb(0,91,150)"  ,
    'medium_airport' :	"rgb(3,57,108)",
    'large_airport':   "rgb(1,31,75)"
    }

    airport_proximity_dict = {
        "large_airport" : 55560,    # 30 nautical miles
        "medium_airport" : 9260,    # 5 nautical miles
        "small_airport" : 5556,     # 3 nautical miles
        "heliport":  2778,          # 1.5 nautical miles
        "seaplane_base" : 5556,     # 3 nautical miles
        "balloonport" : 5556        # 3 nautical miles
    }

    # Plot Trace
    for airport_type in airport_types:

        # Filter by airport type
        airport_df_filtered = airport_df.loc[airport_df['Type'] == airport_type]

        # Airport Marker 
        airport_marker = (go.Scattergeo(
        locationmode = 'USA-states',
        lon = airport_df_filtered['Longitude_Deg'],
        lat = airport_df_filtered['Latitude_Deg'],
        hoverinfo = 'text',
        text = airport_df_filtered.apply(lambda row: f"IATA Code: {row['Iata_Code']}<br>Name: {row['Name']}", axis=1),
        mode = 'markers',
        marker = dict(
            size = 2,
            color = airport_plot_colors[airport_type],
            opacity = 1
            ),
            name = airport_type,
            visible = False
        ))
        all_traces.append(airport_marker)

        # Airport Radius

        lats_plot, lons_plot = [] , []

        for airport in airport_df_filtered.itertuples():
            
            lats, lons = zip(*airport.Airport_Radius)
            lats, lons = list(lats), list(lons)

            lats_plot.extend(lats + [None])
            lons_plot.extend(lons + [None])
        
        airport_radii = (go.Scattergeo(
        locationmode = 'USA-states',
        lon = lons_plot,
        lat = lats_plot,
        hoverinfo = 'skip',
        mode = 'lines',
        line = dict(
            width = 1,
            color = airport_plot_colors[airport_type],
            dash = 'dot'
            ),
            name = airport_type,
            visible = False
        ))
        all_traces.append(airport_radii)




    ## Interactive Buttons 

    hp_buttons = [
            {
                "method": "restyle",
                "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == val]], # When toggled on, checkbox shows already visible traces + haunted place specified in box
                "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == val]], # When toggled off, checkbox removes haunted trace
                "label": val,
                "visible" : True, 

            }
            for val in legend_values
        ]

    hp_toggleAll = {
                "method": "restyle",
                "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name in legend_values]],
                "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name in legend_values]],
                "label": "Toggle All",
                "visible" : True, 

            }
    hp_buttons.append(hp_toggleAll) 


    ## Add Interactive Buttons for airports 

    airport_buttons = [
            {
                "method": "restyle",
                "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == airport_type]],
                "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == airport_type]],
                "label": airport_type,
                "visible" : True, 
            }
            for airport_type in airport_types
        ]

    # Toggle All Airports
    airport_toggleAll = {
                "method": "restyle",
                "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name in airport_types]],
                "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name in airport_types]],
                "label": "Toggle All",
                "visible" : True, 
            }
    airport_buttons.append(airport_toggleAll) 

    # Toggle All Flights 
    flights_toggleAll = {
                "method": "restyle",
                "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == "Flights"]],
                "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == "Flights"]],
                "label": "Flight Paths",
                "visible" : True, 

            }
    airport_buttons.append(flights_toggleAll)



    ## Final Conf 

    updateMenusConf = [
        {
            "buttons": hp_buttons,
            "direction" : "down",
            "showactive" : False,
            "x": 0.1,
            "y": 1.15,
            "xanchor" : "left",
            "yanchor" : "top", 
            "font": {"size" : 12},
            "type": "dropdown",
            "name": "Toggle Flags"
        },
        {
            "buttons": airport_buttons,
            "direction" : "down",
            "showactive" : True,
            "x": -.05,
            "y": 1.15,
            "xanchor" : "left",
            "yanchor" : "top", 
            "font": {"size" : 12},
            "type": "dropdown",
            "name": "Flight Toggle"
        }
        ]


    ## Create Plotly figure 
    fig = go.Figure(data = all_traces)


    fig.update_layout(
        title_text = 'Interactive Flight Map',
        showlegend = True,
        clickmode='event+select',
        hovermode = 'closest',
        geo = dict(
            scope = 'north america',
            projection_type = 'azimuthal equal area',
            showland = True,
            showcountries = True,
            showsubunits = True, 
            subunitcolor = "Black",
            landcolor = 'rgb(243, 243, 243)',
            countrycolor = 'rgb(204, 204, 204)',
        ),
        updatemenus = updateMenusConf
    )

    # Return figure
    fig.show()


# Query
legend_arg = 'Visual_Evidence'
state = None
event_type = None
apparition_type = 'Demon'
haunt_date_range = None
holiday = None

if not legend_arg:
    legend_arg = {'Event_Type': get_legend_items(hp_df, 'Event_Type')} 

legend_arg = {f'{legend_arg}': get_legend_items(hp_df, f'{legend_arg}')} 

additional_tags = ['Haunted_Places_Date']


filtered_hp_df, filtered_route_df, filtered_airport_df = filter_hp_df(
    hp_df,
    route_df,
    airport_df,
    flight_intersections,
    airport_intersections,
    state,
    event_type,
    apparition_type,
    haunt_date_range,
    holiday,
)



hp_interactive_globe(filtered_hp_df, filtered_route_df, filtered_airport_df, legend_arg, additional_tags)



AttributeError: Can only use .str accessor with string values!